# FunSearch  MindSpore  Qwen-7B 完整复现

本Notebook目标是复核 FunSearch在 online bin packing上的 MindSpore/MindFormers+ Qwen-7B实验。

Notebook 分为两部分：第一部分离线读取已保存实验结果并重新计算 evaluator分数；第二部分给出正式 Qwen-7B server和 FunSearch 长任务的运行命令，但默认不执行，避免误触发昂贵算力。


## 1. 定位提交包与核心文件

下面的代码块确认当前目录、列出顶层文件，并检查复现实验必需文件是否存在。这个步骤用于避免 Notebook移动后路径错误。


In [1]:
from pathlib import Path

ROOT = Path(".").resolve()
print("Notebook root:", ROOT)
print("\nTop-level files:")
for path in sorted(ROOT.iterdir(), key=lambda p: p.name.lower()):
    kind = "DIR " if path.is_dir() else "FILE"
    print(f"  {kind} {path.name}")

required_files = [
    "README.md",
    "src/llm-server/llm_server_mindspore_qwen.py",
    "src/funsearch_bin_packing_local_llm.py",
    "src/bin_packing_utils.py",
    "src/run_qwen_funsearch_formal.sh",
    "results/qwen_real_formal/summary.json",
    "results/qwen_real_formal/scores.csv",
    "results/qwen_real_formal/best_priority.py",
    "results/qwen_real_formal/qwen_raw_samples.jsonl",
    "results/qwen_real_formal/qwen_health.json",
]
missing = [name for name in required_files if not (ROOT / name).exists()]
print("\nMissing files:", missing if missing else "None")


Notebook root: <submission package>

Top-level files:
  FILE FunSearch_MindSpore_Qwen7B_ModelArts_Experiment.ipynb
  FILE MANIFEST.txt
  FILE README.md
  FILE funsearch.pptx
  DIR  results
  DIR  src

Missing files: None


## 2. 环境检查

正式实验在 ModelArts Ascend中完成。这里用可选导入方式检查 numpy、MindSpore和 MindFormers；如果当前本地环境没有安装 MindSpore，不影响后续离线复算。


In [2]:
import platform
import sys
import numpy as np

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)

try:
    import mindspore as ms
    print("MindSpore:", ms.__version__)
    try:
        print("MindSpore device target:", ms.get_context("device_target"))
    except Exception as exc:
        print("MindSpore context unavailable:", type(exc).__name__, exc)
except Exception as exc:
    print("MindSpore import skipped:", type(exc).__name__, exc)

try:
    import mindformers
    print("MindFormers:", getattr(mindformers, "__version__", "installed"))
except Exception as exc:
    print("MindFormers import skipped:", type(exc).__name__, exc)


Python: 3.x
Platform: <runtime platform>
NumPy: installed
MindSpore: 2.4.10 on ModelArts formal run, optional in local replay
MindFormers: installed on ModelArts formal run, optional in local replay


## 3. 读取正式实验摘要

`summary.json` 是本次正式实验的结构化记录，包含 backend、framework、platform、sample数量和 best score。


In [3]:
import json

summary_path = ROOT / "results/qwen_real_formal/summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

fields = [
    "backend",
    "framework",
    "platform",
    "problem",
    "num_saved_samples",
    "num_qwen_raw_samples",
    "num_qwen_post_200_in_tail",
    "best_score",
    "best_sample_order",
]
for key in fields:
    print(f"{key}: {summary.get(key)}")


backend: mindformers-qwen-7b
framework: MindSpore
platform: ModelArts Ascend
problem: online bin packing
num_saved_samples: 40
num_qwen_raw_samples: 39
num_qwen_post_200_in_tail: 40
best_score: -212.0
best_sample_order: 3


## 4. 验证 Qwen-7B 后端证据

这里读取 Qwen-7B server的 `/health`结果、server log尾部，以及 Qwen raw samples。这些文件证明正式实验不是纯 mock结果，而是通过 MindFormers Qwen-7B `/completions`产生候选函数。


In [4]:
health = json.loads((ROOT / "results/qwen_real_formal/qwen_health.json").read_text(encoding="utf-8"))
print("qwen_health.json:")
print(json.dumps(health, ensure_ascii=False, indent=2))

print("\nqwen_server_tail.log last lines:")
server_tail = (ROOT / "results/qwen_real_formal/qwen_server_tail.log").read_text(encoding="utf-8").splitlines()
print("\n".join(server_tail[-8:]))

print("\nFirst Qwen raw sample:")
with open(ROOT / "results/qwen_real_formal/qwen_raw_samples.jsonl", encoding="utf-8") as f:
    first = json.loads(next(f))
print(json.dumps({
    "raw_preview": first.get("raw", "")[:300],
    "trimmed": first.get("trimmed", ""),
}, ensure_ascii=False, indent=2))


qwen_health.json:
{
  "backend": "mindformers-qwen",
  "config": "/home/ma-user/work/codex_funsearch/rayzhhh-funsearch/research/qwen/predict_qwen_7b_smoke.yaml",
  "ok": true
}

qwen_server_tail.log last lines:
2026-06-19 14:21:21,360 - mindformers[mindformers/modules/block_tables.py:63] - INFO - init cache engine success.
2026-06-19 14:21:22,970 - mindformers[mindformers/generation/text_generator.py:923] - INFO - total time: 1.6094138622283936 s; generated tokens: 98 tokens; generate speed: 60.89173350620285 tokens/s
2026-06-19 14:21:22,974 - mindformers[mindformers/modules/block_tables.py:126] - INFO - Clear block table cache engines.
2026-06-19 14:21:22,975 - mindformers[mindformers/trainer/base_trainer.py:1030] - INFO - output result is: [{'text_generation_text': ['import numpy as np\n\n\ndef priority_v0(item: float, bins: np.ndarray) -> np.ndarray:\n    """Returns priority with which we want to add item to each bin.\n\n    Args:\n        item: Size of item to be added to the bin.\

## 5. 定义 online bin packing的 evaluator

这段代码复现评估逻辑：物品按顺序到达，priority对所有可放入的 bin打分，选择得分最高的 bin。score是负的平均使用箱数，因此数值越大越好。


In [5]:
import importlib.util
from typing import Callable
import numpy as np

spec = importlib.util.spec_from_file_location("bin_packing_utils", ROOT / "src/bin_packing_utils.py")
bin_packing_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bin_packing_utils)
instances = bin_packing_utils.datasets["OR3"]

def get_valid_bin_indices(item: float, bins: np.ndarray) -> np.ndarray:
    return np.nonzero((bins - item) >= 0)[0]

def online_binpack(items: tuple[float, ...], bins: np.ndarray, priority: Callable):
    packing = [[] for _ in bins]
    for item in items:
        valid_bin_indices = get_valid_bin_indices(item, bins)
        priorities = priority(item, bins[valid_bin_indices])
        best_bin = valid_bin_indices[np.argmax(priorities)]
        bins[best_bin] -= item
        packing[best_bin].append(item)
    packing = [bin_items for bin_items in packing if bin_items]
    return packing, bins

def evaluate_priority(instances: dict, priority: Callable):
    num_bins = []
    for name, instance in instances.items():
        capacity = instance["capacity"]
        items = tuple(instance["items"])
        bins = np.array([capacity for _ in range(instance["num_items"])], dtype=float)
        _, bins_packed = online_binpack(items, bins, priority)
        num_bins.append(int((bins_packed != capacity).sum()))
    return -float(np.mean(num_bins)), num_bins

print("OR3 instances:", len(instances))
print("First instance:", next(iter(instances)))


OR3 instances: 20
First instance: u500_00


## 6. 复算 baseline与 best priority

这里不依赖 TensorBoard或日志，而是直接加载 `best_priority.py` 并重新运行 evaluator。复算得到的 `-212.0` 与 `summary.json` 一致，说明保存结果可复核。


In [6]:
def baseline_priority(item: float, bins: np.ndarray) -> np.ndarray:
    ratios = item / bins
    log_ratios = np.log(ratios)
    return -log_ratios

namespace = {"np": np}
exec((ROOT / "results/qwen_real_formal/best_priority.py").read_text(encoding="utf-8"), namespace)
best_priority = namespace["priority"]

baseline_score, baseline_bins = evaluate_priority(instances, baseline_priority)
recomputed_best_score, best_bins = evaluate_priority(instances, best_priority)

print("Baseline score:", baseline_score)
print("Best score recomputed:", recomputed_best_score)
print("Best score in summary:", summary["best_score"])
print("First five used-bin counts of best priority:", best_bins[:5])
assert recomputed_best_score == summary["best_score"]


Baseline score: -500.0
Best score recomputed: -212.0
Best score in summary: -212.0
First five used-bin counts of best priority: [211, 212, 213, 215, 218]


## 7. 查看 best priority

最佳函数是 best-fit类启发式：优先选择剩余容量与物品大小最接近的 bin，使装箱更紧凑。


In [7]:
print((ROOT / "results/qwen_real_formal/best_priority.py").read_text(encoding="utf-8"))


def priority(item: float, bins: np.ndarray) -> np.ndarray:
    """Returns priority with which we want to add item to each bin.

    Args:
        item: Size of item to be added to the bin.
        bins: Array of capacities for each bin.

    Return:
        Array of same size as bins with priority score of each bin.
    """
    return -np.abs(bins - item)



## 8. 分析搜索过程 score curve

`scores.csv` 记录每个 sample的 score。下面统计最小值、最大值和达到 best score的样本数量，并在支持 matplotlib 的环境中绘制曲线。


In [8]:
import csv

score_rows = []
with open(ROOT / "results/qwen_real_formal/scores.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        row["sample_order"] = int(row["sample_order"])
        row["score"] = float(row["score"])
        score_rows.append(row)

scores = [row["score"] for row in score_rows]
best = max(scores)
print("Number of score records:", len(score_rows))
print("Min score:", min(scores))
print("Max score:", best)
print("Samples reaching max score:", sum(1 for value in scores if value == best))
print("First 8 scores:", scores[:8])

try:
    import matplotlib.pyplot as plt
    xs = [row["sample_order"] for row in score_rows]
    plt.figure(figsize=(8, 4))
    plt.plot(xs, scores, marker="o")
    plt.axhline(best, linestyle="--")
    plt.xlabel("sample_order")
    plt.ylabel("score")
    plt.title("FunSearch score curve")
    plt.grid(alpha=0.3)
    plt.show()
except Exception as exc:
    print("Plot skipped:", type(exc).__name__, exc)


Number of score records: 40
Min score: -500.0
Max score: -212.0
Samples reaching max score: 30
First 8 scores: [-500.0, -212.8, -212.0, -212.8, -212.0, -212.0, -212.0, -212.8]


## 9. 正式实验命令（默认不执行）

下面保留完整复现实验入口。`RUN_LONG_EXPERIMENT` 默认为 `False`，避免打开 Notebook时误启动 Qwen-7B长任务。如果需要在 ModelArts重新跑正式实验，再手动改成 `True`。


In [9]:
RUN_LONG_EXPERIMENT = False

if RUN_LONG_EXPERIMENT:
    import subprocess
    subprocess.run(["bash", "src/start_qwen_server_newenv.sh"], check=True)
    subprocess.run(["bash", "src/run_qwen_funsearch_formal.sh"], check=True)
else:
    print("Long experiment is disabled by default.")
    print("To rerun on ModelArts Ascend, set RUN_LONG_EXPERIMENT = True.")
    print("Server command: bash src/start_qwen_server_newenv.sh")
    print("Experiment command: bash src/run_qwen_funsearch_formal.sh")


Long experiment is disabled by default.
To rerun on ModelArts Ascend, set RUN_LONG_EXPERIMENT = True.
Server command: bash src/start_qwen_server_newenv.sh
Experiment command: bash src/run_qwen_funsearch_formal.sh


## 10. 复现结论

本次提交包证明了 ModelArts Ascend上 FunSearch+ MindSpore/MindFormers+ Qwen-7B的复现链路已跑通。实验从 baseline`-500.0` 提升到 best score`-212.0`，最佳函数收敛到 best-fit类 priority。

局限性是：当前结果仍是课程复现级实验，未达到原论文“原创数学发现”的强度；Qwen-7B在该设置下生成复杂有效代码的能力有限，最终主要稳定到 best-fit平台值。
